# 3. Dimensionality reduction: multi-N anchored alignment <a id="3"></a>
In this section we will apply dimensionality reduction to our coarse-grained representation of selected activation loops.


## Table of contents

- [3.1 Activation loop alignment](#31)
  - [Structural conservation (FoldMason)](#structural-conservation-foldmason)
  - [Multi-N anchored alignment](#multi-n-anchored-alignment)
  - [Run multi-N structure alignments (skippable)](#run-multi-n-structure-alignments-skippable)
  - [Resume from disk (skip re-alignment)](#resume-from-disk-skip-re-alignment)


## Backend map

How this notebook connects to `workflow/` modules (arrows point into the notebook; includes transitive `workflow` subdependencies):

![Backend map](images/backend_maps/04b-MultiNAnchoredAlignment.v2.svg)

<!-- mermaid source (GitHub does not render mermaid in .ipynb; SVG above is for GitHub):
```mermaid
%%{init: {"flowchart": {"nodeSpacing": 12, "rankSpacing": 28, "padding": 4}, "themeVariables": {"fontSize": "11px"}} }%%
flowchart LR
  NB["04b-MultiNAnchoredAlignment.ipynb"]
  m_align["align"]
  m_align_FoldMason["align_FoldMason"]
  m_analyse_alignment_foldmason["analyse_alignment_foldmason"]
  m_chain_basenames["chain_basenames"]
  m_utilities["utilities"]
  m_analyse_alignment_foldmason --> m_align
  m_chain_basenames --> m_utilities
  m_utilities --> m_align
  m_utilities --> m_align_FoldMason
  m_align --> NB
  m_align_FoldMason --> NB
  m_analyse_alignment_foldmason --> NB
  m_utilities --> NB
```
-->


![State of the workflow](images/DimensionalityReduction.png)

To get started, let's load some packages!

In [ ]:
import os
import numpy as np

from workflow.align import Alignment
from workflow.align_FoldMason import AlignmentFoldMason
from workflow.analyse_alignment_foldmason import analyse_alignment
from workflow.utilities import PDBDownloader
from workflow.utilities import (
    count_pdb_files,
    braf_res,
    clear_and_make,
    make_seg,
    copy_filtered_pdbs,
    copy_cg_chain_small_molecules,
)


## 3.1 Activation loop alignment  <a id="31"></a>
The aim of this step is a structurally viable superposition of activation-loop contexts that reduces the impact of missing roto-translational invariance on PCA. Instead of fitting only DFG/APE motif Cα atoms, we align each structure using **multi-N conserved anchors** flanking the loop (plus the DFG/APE motif positions included by the aligner).

### Structural conservation (FoldMason) <a id="structural-conservation-foldmason"></a>
We assess residue conservation with FoldMason (Foldseek structural alphabet → multi-structure alignment). The resulting per-position coverage array (`conservation`) drives anchor selection below. Input structures are the loop-length–filtered chains from curation: `Results/Bounds_CAfilter_chains/`.


In [ ]:
from workflow.align_FoldMason import AlignmentFoldMason

foldmason = AlignmentFoldMason(log_file="multiple_alignment_foldmason.log")

# Multi-structure FoldMason run (template first)
foldmason.process_foldmason_alignment_multi(
    pdb_path="Results/Bounds_CAfilter_chains/",
    target_dir="Results/activation_segments/multi_aligned_foldmason/",
    template_pdb="6UAN_chainD.pdb",
    out_name="msa",
    report_mode=1,
)

In [ ]:
from workflow.analyse_alignment_foldmason import analyse_alignment
import numpy as np
import os

analyser = analyse_alignment()
reference_residues = braf_res()

conservation_run = analyser.run_multi_alignment_conservation_analysis(
    alignment_file="Results/activation_segments/multi_aligned_foldmason/msa_3di.fa",
    reference_name="6UAN_chainD",
    reference_residues=reference_residues,
    conservation_threshold=0.70,
    output_plot="Results/multi_alignment_foldMason_conservation.png",
    output_csv="Results/conserved_residues_70percent.csv",
    show_plot=True,
)

conservation = conservation_run["conservation"]
multi_data = conservation_run["multi_data"]
conserved_df = conservation_run["conserved_df"]

# Cache full conservation vector so later cells can resume without re-alignment
CONSERVATION_NPY = "Results/activation_segments/multi_aligned_foldmason/conservation.npy"
os.makedirs(os.path.dirname(CONSERVATION_NPY), exist_ok=True)
np.save(CONSERVATION_NPY, np.asarray(conservation, dtype=float))
print(f"Saved conservation array: {CONSERVATION_NPY} (len={len(conservation)})")


### Multi-N anchored alignment <a id="multi-n-anchored-alignment"></a>
For each N in 6…26 we take the N nearest ≥90%-conserved positions **before** DFG and **after** APE (non-consecutive walks allowed). The aligner also includes DFG and APE motif Cα positions in every anchor set. Each N produces an independent dataset under `Results/activation_segments/aligned_anchor_{N}/`.


In [ ]:
from workflow.align import Alignment

aligner = Alignment()
reference_pdb = "6UAN_chainD.pdb"

# Multi-anchored alignment (non-consecutive anchors at ≥90% conservation)
# Requires `conservation` and `reference_residues` from the FoldMason cells above.
anchor_info = aligner.compute_anchor_positions_from_conservation(
    reference_pdb=reference_pdb,
    conservation=conservation,
    threshold=0.90,
    n_values=range(6, 27),  # 6..26 inclusive
)

print(f"\nDFG indices (start,end): {anchor_info['dfg_indices']}  (flanks are < dfg_start={anchor_info['dfg_start']})")
print(f"APE indices (start,end): {anchor_info['ape_indices']}  (flanks are >= ape_end={anchor_info['ape_end']})")
print(f"Total positions with ≥90% conservation: {len(anchor_info['conserved_positions'])}")
print(f"Motif positions included in every N set: {anchor_info['motif_positions']}")


### Run multi-N structure alignments (skippable) <a id="run-multi-n-structure-alignments-skippable"></a>

Only needed to **(re)generate** `Results/activation_segments/aligned_anchor_{N}/`. If those folders already exist, skip this cell and use **Resume from disk** below instead.


In [ ]:
pdb_dir = "Results/Bounds_CAfilter_chains/"

anchored_runs = {}
for N, anchors in anchor_info["anchor_sets"].items():
    out_dir = f"Results/activation_segments/aligned_anchor_{N}/"
    print(f"\n=== Anchored alignment with N={N} conserved positions per end (flanks 2N + motifs) ===")

    anchored = aligner.process_anchored_alignment(
        pdb_dir=pdb_dir,
        reference_pdb=reference_pdb,
        output_dir=out_dir,
        anchor_positions=anchors,
        reference_residues=reference_residues,
        matrices_csv=f"{out_dir}/anchored_alignment_matrices_N{N}.csv",
        threshold_label=f"≥90% (N={N} per end; motifs included)",
        ref_name="6UAN_chainD",
    )

    anchored_runs[N] = anchored
    print(f"Anchored alignment kept (N={N}): {len(anchored['kept'])}")
    print(f"Anchored alignment excluded (N={N}): {len(anchored['excluded'])}")


### Resume from disk (skip re-alignment) <a id="resume-from-disk-skip-re-alignment"></a>

Reconstruct `aligner`, `conservation`, `reference_residues`, `anchor_info`, and lightweight `anchored_runs` from cached FoldMason outputs / existing `aligned_anchor_*` folders. Run this after a kernel restart when you only need the N=26 plot, truncation, or PDB counts.


In [ ]:
from workflow.align import Alignment

_ctx = Alignment.restore_multi_n_anchor_context(
    reference_pdb="6UAN_chainD.pdb",
    alignment_file="Results/activation_segments/multi_aligned_foldmason/msa_3di.fa",
    conservation_npy="Results/activation_segments/multi_aligned_foldmason/conservation.npy",
    threshold=0.90,
    n_values=range(6, 27),
)

aligner = _ctx["aligner"]
reference_pdb = _ctx["reference_pdb"]
reference_residues = _ctx["reference_residues"]
conservation = _ctx["conservation"]
anchor_info = _ctx["anchor_info"]
anchored_runs = _ctx["anchored_runs"]

print(f"anchor sets: {sorted(anchor_info['anchor_sets'].keys())}")
print(f"N=26 kept on disk: {len(anchored_runs.get(26, {}).get('kept', []))}")


Plot the N=26 anchor positions on the FoldMason conservation histogram (bordeaux highlights).

In [ ]:
# Conservation plot highlighting anchors used for N=26
N = 26

plot_anchor_info = {
    "left_anchor_positions": list(anchor_info["left_walk"][:N]),
    "right_anchor_positions": list(anchor_info["right_walk"][:N]),
    "motif_positions": list(anchor_info.get("motif_positions") or []),
}

out_dir = f"Results/activation_segments/aligned_anchor_{N}"
out_png = f"{out_dir}/multi_alignment_foldMason_conservation_with_anchors_N{N}.png"

if "anchored_runs" in globals() and N in anchored_runs:
    n_kept = len(anchored_runs.get(N, {}).get("kept", []))
else:
    n_kept = count_pdb_files(out_dir)

aligner.save_conservation_plot_with_anchor_rectangles(
    conservation=conservation,
    reference_residues=reference_residues,
    anchor_info=plot_anchor_info,
    total_structures=n_kept,
    output_file=out_png,
    show_plot=True,
)

print(f"Saved N={N} conservation+anchors plot to: {out_png}")
print(f"Left flanks (N={N}): {plot_anchor_info['left_anchor_positions']}")
print(f"Right flanks (N={N}): {plot_anchor_info['right_anchor_positions']}")


Truncate each aligned-anchor dataset to residues structurally aligned (FoldMason) to BRAF residues 528–697 for visualisation / compact views. Outputs go to `Results/activation_segments/aligned_anchor_trunc_{N}/`. Requires `aligner`, `anchor_info`, and `reference_residues` (from the alignment cell or **Resume from disk**).


In [ ]:
# Copy + truncate aligned structures to residues aligned to BRAF 528–697
summary = aligner.truncate_aligned_anchor_datasets_foldmason(
    anchor_info=anchor_info,
    reference_residues=reference_residues,
    alignment_file="Results/activation_segments/multi_aligned_foldmason/msa_3di.fa",
    reference_id="6UAN_chainD",
    ref_start_resnum=528,
    ref_end_resnum=697,
)

summary

Check how many structures were written for representative N values.

In [ ]:
for N in (6, 26):
    d_align = f"Results/activation_segments/aligned_anchor_{N}/"
    d_trunc = f"Results/activation_segments/aligned_anchor_trunc_{N}/"
    print(
        f"N={N}: aligned={count_pdb_files(d_align)}  "
        f"truncated={count_pdb_files(d_trunc)}"
    )